<a href="https://colab.research.google.com/github/FarazTheAnalyst/LLM-Finetuning/blob/main/LLM-Non-Instruction-Finetuning/non_Instruction_pretrain_llm_finetuning_on_domain_specific_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip install -U peft bitsandbytes transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.4 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 76.8 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.22.0
    Uninstalling transformers-5.22.0:
      Successfully uninstalled transformers-5.22.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bitsandbytes]


In [14]:

!pip install -U trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.6/996.6 kB 6.4 MB/s  0:00:00


In [15]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 MB 38.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [PyMuPDF]


**Prebuilt data from huggingface data hub**

In [16]:
from datasets import Dataset, load_dataset

In [5]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

data/train-00001-of-00004-5852b56a2bd28f(…): downloading bytes:           |  0.00B            

data/train-00002-of-00004-a26307300439e9(…): reconstructing file:   0%|          |  0.00B /  246MB            

... (3 outputs left out)

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [6]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 2119719
})


In [7]:
print(dataset[0])

{'text': 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.\n\nLily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."\n\nTogether, they shared the needle and sewed the button on Lily\'s shirt. It was not difficult for them because they were sharing and helping each other. After they finished, Lily thanked her mom for sharing the needle and fixing her shirt. They both felt happy because they had shared and worked together.'}


In [8]:
print(dataset[1])

{'text': 'Once upon a time, there was a little car named Beep. Beep loved to go fast and play in the sun. Beep was a healthy car because he always had good fuel. Good fuel made Beep happy and strong.\n\nOne day, Beep was driving in the park when he saw a big tree. The tree had many leaves that were falling. Beep liked how the leaves fall and wanted to play with them. Beep drove under the tree and watched the leaves fall on him. He laughed and beeped his horn.\n\nBeep played with the falling leaves all day. When it was time to go home, Beep knew he needed more fuel. He went to the fuel place and got more healthy fuel. Now, Beep was ready to go fast and play again the next day. And Beep lived happily ever after.'}


In [9]:
print(dataset[-1])

{'text': 'Once upon a time, there was an adorable little cat named Kitty. Kitty loved to polish her toy car with a soft cloth. One sunny day, she decided to take her shiny car to the park.\n\nAt the park, she met a friendly dog named'}


**Our own custom data (non instruction data) for domain specific finetuning**

In [17]:
import fitz

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [2]:
def extract_text_from_pdf(pdf_path):
  text_blocks = []
  with fitz.open(pdf_path) as doc:
    for page in doc:
      text = page.get_text("text").strip()
      if text:
        text_blocks.append(text)
  return text_blocks

In [18]:
pdf_texts = extract_text_from_pdf("/content/Metformin.pdf")

In [19]:
pdf_texts

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [20]:
import re
def split_paragraphs(pages):
  paragraphs = []
  for page_text in pages:
    #split on double line breaks or long newlines
    chunks = re.split(r"\n\s*\n", page_text)
    for chunk in chunks:
      clean = chunk.strip()
      if len(clean) > 30: # ignore too short lines
        paragraphs.append(clean)
  return paragraphs

In [21]:
paragraphs = split_paragraphs(pdf_texts)

In [22]:
data = [{"text": p} for p in paragraphs]

In [23]:
print(data)

[{'text': 'Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis.'}, {'text': 'Clinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits he

In [24]:
dataset = Dataset.from_list(data)

In [25]:
dataset

Dataset({
    features: ['text'],
    num_rows: 4
})

**Lets Select the Model**
The checkpoint is from a TinyLLaMA model trained for ~ 1.4M steps over 3 trillion tokens, captured midway before final convergence

In [26]:
# model_name = "meta-llama/llama-2-7b-hf"
# model_name = "meta-llama/llama-3.1-8B"
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"


In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [28]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [29]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [30]:
def tokenize_fn(examples):
  tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=512)
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [31]:
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [32]:
tokenized

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [33]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [34]:
import os
os.makedirs("./llama-pharma-domain", exist_ok=True)

In [35]:
training_args = TrainingArguments(
    output_dir="./llama-pharma-domain",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
)

In [36]:
from transformers import TrainingArguments
help(TrainingArguments)

Help on class TrainingArguments in module transformers.training_args:

class TrainingArguments(builtins.object)
 |  TrainingArguments(
 |      output_dir: str | None = None,
 |      per_device_train_batch_size: int = 8,
 |      num_train_epochs: float = 3.0,
 |      max_steps: int = -1,
 |      learning_rate: float = 5e-05,
 |      lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear',
 |      lr_scheduler_kwargs: dict | str | None = None,
 |      warmup_steps: float = 0,
 |      optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused',
 |      optim_args: str | None = None,
 |      weight_decay: float = 0.0,
 |      adam_beta1: float = 0.9,
 |      adam_beta2: float = 0.999,
 |      adam_epsilon: float = 1e-08,
 |      optim_target_modules: None | str | list[str] = None,
 |      gradient_accumulation_steps: int = 1,
 |      average_tokens_across_devices: bool = True,
 |      max_grad_norm: float = 1.0,
 |      label_smoothing_factor: float = 

In [37]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized
)

In [34]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 14.28 GiB is allocated by PyTorch, and 146.96 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)


 **Here we are not specfiying anything means this is full fine-tuning
Now we have two methods for Partial Finetuning**

1. Freeze some layer and finetune unfreeze layer(old CNN nad Bert style method)
2. LORA(Append some external weight to the already trained pretrain weight)

3. First Let's see the method where we are going to freeze the layer


In [35]:
# 1 install dependencies

In [36]:
# pip install -q transformers datasets accelerate bitsandbytes

In [37]:
# 2 imports

In [38]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
# from datasets import load_dataset
# import os, re

In [39]:
# 3 load base model and tokenizer

In [40]:
# model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

In [41]:
# 4 Freeze everything
# for param in model.parameters():
#   param.requires_grad = False

In [42]:
# 5 Unfreeze last 4 transformer blocks + lm_head

In [43]:
# for name, param in model.named_parameters():
#   if any(f"layers.{i}." in name for i in range(20, 24)): # last 4 layers
#        param.requires_grad = True
#   if "lm_head" in name:
#       param.requries_grad = True

# Verify how many parameters are trainable
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total = sum(p.numel() for p in model.parameters())
# print(f"✅ Trainable: {trainable/total*100:.2f}% of total paramters")

In [44]:
# 6 load your dataset

In [45]:
# dataset = load_dataset("json", data_files={"train": "pharma_non_instrution.jsonl"})

# def tokenize_fn(examples):
#      tokens = tokenizer(
#          examples["text"],
#          truncation=True,
#          padding="max_length",
#          max_length=512
#      )
#      tokens["labels"] = tokens["input_ids"].copy()
#      return tokens

# tokenize = dataset["train"].map(tokenize_fn, batched=True, remove_columns["text"])


In [46]:
# 7 Data collator for causal LM

In [47]:
# data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

In [48]:
# 8 Training arguments

In [49]:
# os.environ["WANDB_DISABLED"] = "true"  # disable wandb
# training_args = TrainingArguments(
#     output_dir="./tinyllama-pharma-last4",
#     num_train_epochs=2,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=1e-4,        # slightly higher since fewer params
#     fp16=True,
#     logging_steps=20,
#     save_steps=200,
#     save_total_limit=2,
#     report_to="none"
# )


In [50]:
# Trainer setup

In [51]:
# trainer = Trainer(
#    model=model,
#    args=training_args,
#    train_dataset=tokenized,
#    data_collator=data_collator
#    )

In [52]:
# 10 Start training

In [53]:
# trainer.train()

In [54]:
# trainer.save_model("./tinyllama-pharma-last4-final")
# print("\n✅ Training completed and model saved")

Now lets see the **LORA** based method

In [38]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [39]:
!pip install -U peft bitsandbytes transformers accelerate

In [40]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [41]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [42]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [43]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [44]:
def tokenize_fn(examples):
  tokens = tokenizer(
      examples["text"],
      truncation=True,
      padding="max_length",
      max_length=512
  )
  tokens["labels"] = tokens["input_ids"].copy()
  return tokens

In [45]:
tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [46]:
tokenized

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 4
})

In [47]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [48]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [49]:
from peft import prepare_model_for_kbit_training

In [50]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

In [51]:
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(model)

In [52]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [53]:
# save adapter

non_inst_model_lora = get_peft_model(model, lora_config)

In [54]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none"
)

In [55]:
trainer = Trainer(
    model=non_inst_model_lora,
    args=args,
    train_dataset=tokenized
)

In [56]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=3, training_loss=10.10820198059082, metrics={'train_runtime': 6.292, 'train_samples_per_second': 1.907, 'train_steps_per_second': 0.477, 'total_flos': 38177788133376.0, 'train_loss': 10.10820198059082, 'epoch': 3.0})

In [62]:
model_path = "/content/tinyllama-lora/checkpoint-3"

In [63]:
# tokenizer = AutoTokenizer.from_pretrained(model_path)

In [66]:
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

In [69]:
prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

In [70]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [73]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [78]:
print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))


Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe resulted in the greatest protection against cardiovascular death. In addition, ANGIO-13, a novel 24-hour follow up study in 2,058 patients undergoing coronary artery bypass surgery showed that treatment with atorvastatin with ezetimibe lowered the risk of angina pectoris re-occurence and myocardial infarction compared to placebo.
Lipitor (atorv


**Non-Instruction FineTuning Dataset**

In [ ]:

dataset = load_dataset("HuggingFaceFW/fineweb")
pubmed = load_dataset("ncbi/pubmed")
dataset = load_dataset("datajuicer/the-pile-pubmed-abstracts-refined-by-data-juicer")
dataset = load_dataset("open-llm-leaderboard/open_llm_corpus")
owt = load_dataset("Skylion007/openwebtext")
ds = load_dataset("armanc/scientific_papers")

**Instruction Finetuning Dataset**

In [ ]:
https://huggingface.co/datasets/Amod/mental_health_counseling_conversations
https://huggingface.co/datasets/yahma/alpaca-cleaned
https://huggingface.co/datasets/Open-Orca/OpenOrca
https://huggingface.co/datasets/tatsu-lab/alpaca
https://huggingface.co/datasets/OpenAssistant/oasst1

Preference Alignement Tuning Datset

In [ ]:
https://huggingface.co/datasets/Anthropic/hh-rlhf
https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences-cleaned
https://huggingface.co/datasets/xinlai/Math-Step-DPO-10K